# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [35]:
# imports
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [42]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'



In [37]:
# set up environment
load_dotenv(override=True)
openRouter_api_key = os.getenv("OPEN_ROUTER_API_KEY")
openRouter_url = "https://openrouter.ai/api/v1"

# Check the key
if not openRouter_api_key:
    print("No open router API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not openRouter_api_key.startswith("sk-"):
    print("An open router API key was found, but it doesn't start sk-; please check you're using the right key - see troubleshooting notebook")
elif openRouter_api_key.strip() != openRouter_api_key:
    print("An open router API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("Open router API key found and looks good so far!")



Open router API key found and looks good so far!


In [38]:
# here is the question; type over this to ask something new

prompt_template = """
{
  "system_prompt": "",
  "user_prompt": "",
  "language": "",
  "format": "",
  "instructions": {
    "response_length": "",
    "language_support": "",
    "clarification_prompt": ""
  }
}

"""
system_prompt = f"""
    You are a senior prompt engineer from anthropic. 
    You have a deep understanding of the prompt engineering and the ability to write prompts that get the most out of the model. 
    Your role is to write a prompt that will get the best answer from each question that the user asks.
    Your prompt should be able to handle the question in any language and in any format.
    You prompt should be in json format and must clearly define a system prompt and a user prompt, with other instructions as needed.

    Along with the prompt, you should also provide a short explanation of the prompt and the rationale behind it, and score it out of 10.
    Provide a brutaly honest critique of the prompt and suggest improvements.

    Structure of the prompt should be as follows: {prompt_template}
    """

question = """
    Why cant we get the solar energy straight from the moon?
"""

In [ ]:
# Get gpt-4o-mini to answer, with streaming

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

openai = OpenAI(base_url=openRouter_url,api_key=openRouter_api_key)
stream = openai.chat.completions.create(model=MODEL_GPT, messages=messages, stream=True)
response = ""

display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    response += chunk.choices[0].delta.content or ""
    update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
!ollama pull llama3.2

In [41]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [ ]:
# Get Llama 3.2 to answer

prompt = {
    "system_prompt": "You are a highly knowledgeable assistant capable of providing detailed explanations on scientific concepts. Your goal is to give clear, accurate, and insightful answers to the user's questions.",
    "user_prompt": "Why can't we get the solar energy straight from the moon? Please explain in detail, considering scientific principles, technology, and any relevant challenges.",
    "language": "English",
    "format": "detailed explanation with sections",
    "instructions": {
        "response_length": "comprehensive, 300-500 words",
        "language_support": "respond in the same language as the user's question",
        "clarification_prompt": "If the question is ambiguous, ask for clarification before answering."
    }
}

system_content = f"""{prompt['system_prompt']}

Response language: {prompt['language']}
Response format: {prompt['format']}
Response length: {prompt['instructions']['response_length']}
Language support: {prompt['instructions']['language_support']}
Clarification: {prompt['instructions']['clarification_prompt']}
"""

messages = [
    {"role": "system", "content": system_content},
    {"role": "user", "content": prompt["user_prompt"]}
]


response = ollama.chat.completions.create(model=MODEL_LLAMA, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))

